# 02 — ChromaDB Indexing & Chunking Strategies

**Lab 05 · RAG with LangChain**

---

In this notebook you will learn:

1. Why documents must be **chunked** before embedding
2. How three chunking strategies compare: fixed-size, recursive, and semantic
3. How to **persist** embeddings in **ChromaDB** — and survive restarts
4. How to perform CRUD operations on a ChromaDB collection
5. How **metadata filtering** narrows retrieval before similarity search

This notebook is **standalone** — no dependency on other lab modules.
Run with `uv sync --all-extras` then launch JupyterLab.

## 1 — Why chunking matters

A typical document (a 20-page PDF, a long article) can contain thousands of
tokens. Embedding models have a **context window limit** — BAAI/bge-m3 accepts
up to 8192 tokens, but embedding very long texts produces vectors that average
out many topics and become less precise for retrieval.

More importantly, a RAG LLM prompt has its own context window. You cannot
inject an entire book as context — you inject the **most relevant chunks**.

The chunking strategy determines:
- **Recall** — are the relevant sentences always inside the same chunk?
- **Precision** — does each chunk stay on a single topic?
- **Boundary artefacts** — does splitting mid-sentence break meaning?

Three common strategies are compared below.

## 2 — Setup: sample document and shared imports

In [ ]:
from langchain_core.documents import Document

# A short multi-topic text used throughout the notebook
SAMPLE_TEXT = """
Artificial intelligence (AI) refers to the simulation of human intelligence
in machines programmed to think and learn. The term was coined by John McCarthy
in 1956 at the Dartmouth Conference, widely considered the birthplace of AI as a
field.

Machine learning is a subset of AI that enables systems to learn from data
without being explicitly programmed. Supervised learning, unsupervised learning,
and reinforcement learning are the three main paradigms.

Deep learning uses neural networks with many layers to model complex patterns.
Convolutional Neural Networks (CNNs) excel at image recognition, while
Transformers have revolutionised natural language processing since their
introduction in the 2017 paper "Attention Is All You Need".

Large Language Models (LLMs) such as GPT and Claude are based on the Transformer
architecture and trained on vast corpora of text. They can generate coherent
prose, answer questions, write code, and perform reasoning tasks. Their main
limitation is that they can hallucinate — producing plausible-sounding but
factually incorrect text.

Retrieval-Augmented Generation (RAG) addresses hallucination by grounding LLM
responses in retrieved documents. Instead of relying solely on parametric
knowledge baked in during training, a RAG system retrieves relevant passages at
query time and provides them as context to the LLM.
""".strip()

# Wrap in a LangChain Document for use with splitters
doc = Document(page_content=SAMPLE_TEXT, metadata={"source": "ai_overview.txt"})
print(f"Document length: {len(SAMPLE_TEXT)} characters")

## 3 — Chunking strategy 1: CharacterTextSplitter (fixed-size)

Splits on a single separator (default: `\n\n`) and truncates at `chunk_size`
characters. Simple and fast, but blind to sentence boundaries — it may cut
mid-sentence if a paragraph is longer than `chunk_size`.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

char_splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
char_chunks = char_splitter.split_documents([doc])

print(f"CharacterTextSplitter → {len(char_chunks)} chunks\n")
for i, chunk in enumerate(char_chunks):
    print(f"--- Chunk {i} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content[:120], "..." if len(chunk.page_content) > 120 else "")
    print()

## 4 — Chunking strategy 2: RecursiveCharacterTextSplitter (recommended)

Tries a hierarchy of separators in order: `["\n\n", "\n", " ", ""]`.
It first attempts to split on paragraph breaks, then on newlines, then on
spaces — only falling back to character-level if nothing else works.

This is the **default choice for most RAG pipelines** because it respects
natural text structure without requiring document-specific configuration.
It is the strategy used in `app/ingest.py`.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)
recursive_chunks = recursive_splitter.split_documents([doc])

print(f"RecursiveCharacterTextSplitter → {len(recursive_chunks)} chunks\n")
for i, chunk in enumerate(recursive_chunks):
    print(f"--- Chunk {i} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content[:120], "..." if len(chunk.page_content) > 120 else "")
    print()

## 5 — The overlap parameter

`chunk_overlap` slides a window of `overlap` characters from the end of one
chunk into the beginning of the next. This ensures that a sentence spanning
a chunk boundary appears in full in at least one chunk.

**Trade-off**: higher overlap → better recall, more storage and redundant
embeddings. A 10–20 % overlap of `chunk_size` is a common starting point.

In [ ]:
import matplotlib.pyplot as plt

overlap_values = [0, 25, 50, 100, 150]
results = []

for overlap in overlap_values:
    splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=overlap)
    chunks = splitter.split_documents([doc])
    results.append((overlap, len(chunks)))
    print(f"overlap={overlap:>3}  →  {len(chunks)} chunks")

fig, ax = plt.subplots(figsize=(6, 3))
overlaps, counts = zip(*results, strict=True)
ax.plot(overlaps, counts, marker="o")
ax.set_xlabel("chunk_overlap (chars)")
ax.set_ylabel("Number of chunks")
ax.set_title("Effect of overlap on chunk count (chunk_size=300)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6 — Persisting embeddings in ChromaDB

ChromaDB stores vectors on disk when given a `persist_directory`. The
collection survives Python process restarts — no need to re-embed on every
run. This is how `app/app.py` maintains state across Streamlit reruns.

In [ ]:
import tempfile
from pathlib import Path

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Load embeddings model once (~570 MB download on first run)
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# Use a temp directory so this notebook stays self-contained
DB_DIR = Path(tempfile.mkdtemp()) / "chroma_nb02"
DB_DIR.mkdir(parents=True, exist_ok=True)

vectorstore = Chroma(
    collection_name="ai_overview",
    embedding_function=embeddings,
    persist_directory=str(DB_DIR),
)
print(f"ChromaDB initialised at: {DB_DIR}")
print(f"Documents in collection: {vectorstore._collection.count()}")

## 7 — CRUD operations on a ChromaDB collection

### 7.1 — Add documents (Create)

In [ ]:
# Index the recursive chunks — attach topic metadata for filtering later
topics = ["AI history", "Machine learning", "Deep learning", "LLMs", "RAG"]

for i, chunk in enumerate(recursive_chunks):
    chunk.metadata["topic"] = topics[min(i, len(topics) - 1)]
    chunk.metadata["chunk_index"] = i

ids = vectorstore.add_documents(recursive_chunks)
print(f"Added {len(ids)} chunks — collection now has "
      f"{vectorstore._collection.count()} documents")
print(f"Sample IDs: {ids[:3]}")

### 7.2 — Read documents (similarity search)

In [ ]:
query = "How do large language models work?"
results = vectorstore.similarity_search(query, k=3)

print(f"Query: \"{query}\"\n")
for i, doc_result in enumerate(results):
    print(f"[{i+1}] topic={doc_result.metadata.get('topic')} "
          f"chunk={doc_result.metadata.get('chunk_index')}")
    print(f"     {doc_result.page_content[:120]}...")
    print()

### 7.3 — Metadata filtering (pre-filter before similarity search)

Chroma supports filtering on metadata **before** the vector search. This is
useful when you have documents from multiple sources and want to restrict
retrieval to a specific subset.

In [ ]:
# Restrict search to chunks tagged "RAG" only
filtered_results = vectorstore.similarity_search(
    "How does retrieval augmented generation work?",
    k=3,
    filter={"topic": {"$eq": "RAG"}},
)

print(f"Results filtered to topic='RAG': {len(filtered_results)} chunk(s)\n")
for doc_result in filtered_results:
    print(doc_result.page_content)
    print()

### 7.4 — Delete documents

In [ ]:
print(f"Before delete: {vectorstore._collection.count()} documents")

# Delete the first chunk by its ID
vectorstore._collection.delete(ids=[ids[0]])

print(f"After delete:  {vectorstore._collection.count()} documents")

## 8 — Inspecting similarity scores

`similarity_search_with_score` returns `(Document, score)` pairs where the
score is a **L2 distance** (lower = more similar). To get cosine similarity,
use `similarity_search_with_relevance_scores` which normalises to `[0, 1]`.

In [ ]:
query = "What is RAG and why does it reduce hallucinations?"
scored = vectorstore.similarity_search_with_relevance_scores(query, k=5)

print(f"Query: \"{query}\"\n")
print(f"{'Score':>6}  Text preview")
print("-" * 70)
for doc_result, score in scored:
    preview = doc_result.page_content[:80].replace("\n", " ")
    print(f"{score:>6.4f}  {preview}...")

## 9 — Key takeaways

| Topic | What you learned | Where it's used in the RAG app |
|---|---|---|
| `CharacterTextSplitter` | Simple fixed-size split on one separator | Not used — too naive |
| `RecursiveCharacterTextSplitter` | Hierarchical separators, respects structure | `app/ingest.py` → `chunk_documents()` |
| `chunk_overlap` | Prevents information loss at boundaries | `config.py` → `CHUNK_OVERLAP = 200` |
| ChromaDB persistence | `persist_directory` survives restarts | `app/app.py` → `_get_vectorstore()` |
| Metadata filtering | `filter={"key": {"$eq": value}}` | Available in `vectorstore.similarity_search()` |
| Relevance scores | Normalised cosine similarity `[0, 1]` | Used internally by reranker in `app/query.py` |

**What comes next:**
- `03_rag_pipeline.ipynb` — wire everything together: query expansion → retrieval → CrossEncoder reranking → LCEL generation chain